# HumorVibes — the Humor Genome, wave 2

### Humor as affordable surprise, measured with Gemma, over a corpus grown from 23,885 items to 3,164,600

This notebook is the write-up. Theory, method, data provenance, results and limits are here, next to the code that produces every number quoted.

---

## The theory

Starting point: Friston's account of the brain as a system minimising surprise. The hypothesis this project tests is that **a joke is a controlled prediction error with a cheap, permitted repair.**

- The setup commits the listener to a dominant reading.
- The punchline is low-probability under that reading — **surprise, S**.
- A hidden frame exists under which it snaps into place — **resolution, R**.
- The re-route has to be affordable (**efficiency, E**: one line of frame, not a paragraph) and permitted (**bad surprise, B**).

B is not a synonym for *offensive*. It is audience-relative, so every check is persona-conditioned. The controlling definition is kept verbatim throughout:

> Bad surprise is poorly defined, a bad surprise is a surprise that contradicts with internal
> models within a human brain that are so strong they override logic and are some of the primary
> drivers of a person's perception, understanding, and good/bad/moral/ethical views of the world.
> So basically, a surprise is not good if it disagrees with something that is already overriding
> logic or a surprise is not good it if disagrees with a nearly overwhelming generalization engine
> in a human mind that has significant overriding power to override logic, promote other false
> generalizations, and is the primary feature used to reduce surprise in that person's mind.

Each failure mode becomes diagnosable: predictable (low S), nonsense (high S, no frame), dissected frog (frame too expensive), offence (B collision).

## Gemma as instrument, not oracle

A causal language model is itself a predictive system, so this project never asks Gemma to *rate* how funny something is. It reads surprise off the logits, teacher-forced:

**S = mean NLL(punchline tokens | setup)** — one deterministic forward pass, no sampling, no temperature, no rubric.

That distinction is the whole method. A rating is an opinion the model was trained to produce; a log-probability is a measurement of the model's own expectation. Only the second one can falsify anything.

The instrument is checked against a value pinned across earlier runs — **S = 3.19** on a fixed reference joke — before it is used for anything new. Four independent instruments have agreed on it: llama.cpp at Q4 and Q8, in-kernel transformers, and a quantisation cross-check. If the check below disagrees, nothing after it should be believed.

In [ ]:
import subprocess, sys, os, json, textwrap, glob, hashlib
REPO = "https://github.com/aidonerightcorp/humorvibes-jestry"
REPO_REF = "humor-genome-wave2-v3"
if not os.path.exists('humorvibes-jestry'):
    subprocess.run(['git','clone','--depth','1','--branch',REPO_REF,REPO,
                    'humorvibes-jestry'], check=True)
sys.path.insert(0, 'humorvibes-jestry')
os.chdir('humorvibes-jestry')
commit = subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip()
print('repo at', os.getcwd())
print('pinned source', REPO_REF, commit)

In [ ]:
# The full corpus is ~1 GB and lives outside git, so this reads a stratified
# slice attached as a Kaggle dataset; falls back to the repo corpus locally.
from pathlib import Path
# FIND the dataset by its contents, not by a guessed mount path. Kaggle
# nests mounts (the model turned up under /kaggle/input/models/...), and an
# earlier run guessed wrong, silently fell through to the repo's own small
# corpora/ directory, and reported a census of the OLD corpus as though it
# were the new one. A fallback that succeeds with the wrong data is worse
# than one that fails.
hits = glob.glob('/kaggle/input/**/corpus_sample.jsonl', recursive=True)
DATA_DIR = os.path.dirname(hits[0]) if hits else None
if DATA_DIR is None:
    for d in ('kaggle_wave2', 'corpora'):
        if os.path.isdir(d) and glob.glob(os.path.join(d, '*.jsonl')):
            DATA_DIR = d
            print(f'WARNING: wave-2 dataset not mounted; falling back to {d!r}. '
                  'Counts below describe THAT corpus, not the published slice.')
            break
assert DATA_DIR, 'no corpus found — attach taylorsamarel/humor-genome-wave2'
print('corpus dir:', DATA_DIR)
for p in sorted(glob.glob(os.path.join(DATA_DIR, '*'))):
    print(f'  {os.path.basename(p):<42} {os.path.getsize(p):>12,} B')

# Verify the bytes Kaggle mounted before deriving any claim from them.
manifest_path = os.path.join(DATA_DIR, 'manifest.json')
assert os.path.exists(manifest_path), 'dataset manifest is missing'
manifest = json.load(open(manifest_path, encoding='utf-8'))
for name, evidence in manifest.items():
    path = os.path.join(DATA_DIR, name)
    assert os.path.exists(path), f'manifest payload missing: {name}'
    digest = hashlib.sha256()
    with open(path, 'rb') as fh:
        while chunk := fh.read(1024 * 1024):
            digest.update(chunk)
    assert os.path.getsize(path) == evidence['bytes'], f'byte length mismatch: {name}'
    assert digest.hexdigest() == evidence['sha256'], f'sha256 mismatch: {name}'
print(f'manifest verified: {len(manifest)} payload files')
FULL_CENSUS = json.load(open(os.path.join(DATA_DIR, 'census.json'), encoding='utf-8'))
EXPORT_SUMMARY = json.load(open(os.path.join(DATA_DIR, 'export_summary.json'), encoding='utf-8'))
assert FULL_CENSUS['items'] == EXPORT_SUMMARY['full_corpus_rows']

import style_taxonomy as st, corpus_census as cc
import verify_wave2_release as release_gate
release_receipt = release_gate.verify(Path(DATA_DIR))
print('semantic release gate:', release_receipt['status'], f"({release_receipt['exported_rows']:,} rows)")
# Point at the CORPUS file only. Both modules glob CORPORA/*.jsonl, and the
# export ships derived sidecars (frames, aligned phrases) in the same folder
# with a different schema — globbing the folder silently mixed them in.
CORPUS_FILES = [Path(p) for p in sorted(glob.glob(os.path.join(DATA_DIR, 'corpus_sample.jsonl')))]
if not CORPUS_FILES:
    CORPUS_FILES = [Path(p) for p in sorted(glob.glob(os.path.join(DATA_DIR, '*.jsonl')))]
_orig_iter = st.iter_corpus
st.iter_corpus = lambda paths=None, strict=False: _orig_iter(paths or CORPUS_FILES, strict=strict)
print('\ncorpus files:', [p.name for p in CORPUS_FILES])
print('rows visible:', sum(1 for _ in st.iter_corpus()))

## 1. What is actually in the corpus

Wave 2 was a six-lane sourcing sweep: HuggingFace datasets, keyless APIs, multilingual phrase and translation sources, memes, academic benchmarks, and joke styles. 144 HF datasets, 78 API endpoints, 301 Wikiquote pages across 51 language editions and 164 Wiktionary categories were verified live, each with a status code and a row count.

The census groups sources into **families** before measuring concentration. Without that it lies by dilution: the caption archive writes one source string per contest, so 385 sibling sources each look tiny while together they dominate.

In [ ]:
c = FULL_CENSUS
sample_c = cc.census(CORPUS_FILES)
assert sample_c['items'] == EXPORT_SUMMARY['exported_rows']
print('FULL CORPUS')
print(f"items {c['items']:,}   sources {c['distinct_sources']}   "
      f"languages {c['distinct_languages']}   licences {c['distinct_licences']}")
print(f"graded (carry a human funniness signal): {c['graded']:,} = {c['graded_share']:.1%}")
ts, tn = c['top_source']
print(f"\nlargest source family: {ts}\n  {tn:,} rows = {c['top_source_share']:.1%}")
print('\nlicence classes:')
for k, v in sorted(c['licence_classes'].items(), key=lambda kv: -kv[1]):
    print(f'  {k:<18} {v:>9,}  {v/c["items"]:>6.1%}')
print('\ntop source families:')
for k, v in list(c['families'].items())[:12]:
    print(f'  {k[:56]:<56} {v:>9,}')
print(f"\nPUBLISHED STRATIFIED SLICE: {sample_c['items']:,} rows; "
      f"largest-family share {sample_c['top_source_share']:.1%}")

### Languages

One bug here is worth stating because it was invisible rather than loud. The content screen required a record to be at least **8 characters**. A Chinese *chengyu* is **four** (一箭双雕). Of 16,920 Chinese idioms served by Wiktionary, only 1,629 survived — an English-shaped length threshold was deleting most CJK content while looking like a safety check. After making the floor script-aware, Chinese went to **18,384** and became the largest language in that lane.

This is the failure mode worth designing against: a filter that removes real data and reports nothing.

In [ ]:
langs = c['languages']; tot = sum(langs.values()); mx = max(langs.values())
for k, v in list(langs.items())[:26]:
    print(f'  {k:<9} {v:>8,} {v/tot:>6.2%}  ' + '#' * max(1, int(52 * v / mx)))

## 2. Styles of joke

The corpus began with **no style axis at all**, so it could not answer the most natural question anyone asks about humor: does a military joke work differently from a dad joke?

Three independent axes now exist:

| axis | what it is | how it is assigned |
|---|---|---|
| **FORM** | the structural template — what shape the expectation-and-turn takes | 41 regex templates, most-specific-first |
| **DOMAIN** | what the joke is *about* | 25 keyword lexicons, reported as a guess |
| **STYLE_CATEGORY** | declared by the source itself | curated public-domain volumes and category APIs |

The third is the strongest, because the label does not have to be inferred. Project Gutenberg publishes single-subject humor volumes — a book of army jokes is a book of army jokes — and APIs like Chuck Norris and JokeAPI ship a category on every item.

Form templates now include 13 **non-English native forms** (German *Treffen sich zwei*, Russian *Штирлиц* and *Вовочка*, French *Monsieur et Madame*, Portuguese *O que é*, Chinese *xiehouyu*, Japanese *dajare*, Korean *ajae-gag*, and others). Before those existed, non-English specific-form coverage measured approximately **zero** — every non-English item fell into a generic bucket, and the taxonomy silently described only English humor while reporting high coverage.

In [ ]:
print('self-test on the form classifier:')
rc = st.selftest()
assert rc == 0, 'form classifier self-test failed'
print(f'\n{len(st.FORM_RULES)} form templates (+ a rhyme-verified limerick check), '
      f'{len(st.DOMAIN_LEXICON)} domains')

In [ ]:
from collections import Counter
rows = list(st.iter_corpus())
labels = [st.label_item(r) for r in rows]
n = len(labels)
forms = Counter(l['form'] for l in labels)
print(f'{n:,} items labelled\n')
print('FORM — specific templates (the informative ones):')
for k, v in forms.most_common():
    if k in st.GENERIC_FORMS or k == 'unknown':
        continue
    print(f'  {k:<24} {v:>7,}  {v/n:>6.3%}')
spec = sum(v for k, v in forms.items() if k not in st.GENERIC_FORMS and k != 'unknown')
print(f'\nspecific-form share: {spec:,} = {spec/n:.1%}')
print('generic buckets (shape, not mechanism):')
for k in list(st.GENERIC_FORMS) + ['unknown']:
    if forms.get(k):
        print(f'  {k:<24} {forms[k]:>7,}  {forms[k]/n:>6.1%}')

### Occupational and national styles

`style_category` comes from the source, not from a classifier. These are curated single-subject public-domain volumes — military, medical, legal, Jewish, Russian, German, Irish, Scottish, Dutch, Italian, Finnish, Nasreddin, plus form-specific collections for clerihews, spoonerisms, epigrams, epitaphs, riddles and wellerisms.

Two of those volumes were **excluded by id rather than by keyword**: minstrel joke books whose slurs are written in dialect spelling that the screening regex does not catch. A keyword filter would have passed them.

In [ ]:
cats = Counter((r.get('meta') or {}).get('style_category')
               for r in rows if (r.get('meta') or {}).get('style_category'))
declared = Counter((r.get('meta') or {}).get('style')
                   for r in rows if (r.get('meta') or {}).get('style'))
print('style_category — declared by a curated source:')
for k, v in cats.most_common():
    print(f'  {k:<22} {v:>7,}')
print('\nstyle — declared by a category API (Chuck Norris, JokeAPI, dad-joke search):')
for k, v in declared.most_common(14):
    print(f'  {k:<22} {v:>7,}')

In [ ]:
domains = Counter(l['domain'] for l in labels)
print('DOMAIN — lexical guess, including branch-level military:')
for k, v in domains.most_common():
    print(f'  {k:<18} {v:>7,}  {v/n:>6.2%}')

In [ ]:
cov = st.coverage(labels)
print(f"{'lang':<8} {'n':>8} {'any-form':>9} {'specific':>9}")
for lang, v in list(cov.items())[:18]:
    print(f"{lang:<8} {v['n']:>8,} {v['share']:>8.1%} {v['specific_share']:>9.1%}")
print('\n`any-form` counts generic buckets and flatters every language.')
print('`specific` is the honest column.')

## 3. Expectation and violation, annotated apart

For 705 New Yorker cartoons, three crowd workers each wrote **two separate fields**: what the scene *is*, and what is *uncanny* about it. That is a stated expectation and a stated violation, annotated independently, before any caption exists. A further 647 captions carry a free-text explanation of why they work — the repair, in words.

Nothing else found in the whole sweep separates the three parts of a joke this explicitly, which is why these are stamped as frame-carrying supply rather than as ordinary text.

In [ ]:
def read_jsonl(path):
    rows_ = []
    if not os.path.exists(path):
        return rows_
    with open(path, encoding='utf-8') as fh:
        for line in fh:
            r = json.loads(line)
            if '_meta' not in r:
                rows_.append(r)
    return rows_

frames = read_jsonl(os.path.join(DATA_DIR, 'expectation_violation_frames.jsonl'))
if not frames:
    for r in st.iter_corpus():
        m = r.get('meta', {}) or {}
        if m.get('image_uncanny_description'):
            frames.append({'contest': m.get('contest'),
                           'expectation': m.get('image_description', ''),
                           'violation': m['image_uncanny_description'],
                           'caption': r['text'],
                           'explanation': m.get('explanation', '')})
print(f'{len(frames):,} rows carrying an annotated expectation/violation pair\n')
seen = set()
for f in frames:
    if f['contest'] in seen or not f.get('violation'):
        continue
    seen.add(f['contest'])
    print(f"--- contest {f['contest']}")
    print(textwrap.fill('EXPECTATION: ' + f['expectation'], 94))
    print(textwrap.fill('VIOLATION:   ' + f['violation'], 94))
    print()
    if len(seen) >= 3:
        break

In [ ]:
expl = [f for f in frames if f.get('explanation')]
print(f'{len(expl):,} captions carry a human explanation of the repair\n')
for f in expl[:2]:
    print(textwrap.fill('CAPTION:     ' + f['caption'], 94))
    print(textwrap.fill('EXPLANATION: ' + f['explanation'], 94))
    print()

### Translation-aligned phrases

Cross-lingual claims need pairs, not two monolingual piles. An 1869 public-domain polyglot supplies proverbs in seven languages *each with its English translation on the same line*, and several HuggingFace sets add parallel Yoruba, Urdu and Chinese.

In [ ]:
aligned = read_jsonl(os.path.join(DATA_DIR, 'aligned_phrases.jsonl'))
print(f'{len(aligned):,} aligned pairs across '
      f"{len(set(a['language'] for a in aligned))} languages\n")
for a in aligned[:5]:
    print(f"  [{a['language']}] {a['text'][:56]}")
    print(f"        -> {a['translation_en'][:64]}")

## 4. Measuring with Gemma

Everything above counts. This is where a model is asked a question.

The instrument check comes first, and it gates everything below it.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# DISCOVER the checkpoint, never hard-code the mount. A hard-coded path
# failed here once already: transformers sees a non-existent directory,
# assumes it is a hub repo id, and raises a confusing 'Repo id must be in
# the form namespace/repo_name'. The repo already solved this once, so reuse
# its finder and fall back to a glob.
from mesh_signals import TransformersProvider
MODEL = TransformersProvider._find_kaggle_gemma()
if not MODEL:
    hits = [d for d in glob.glob('/kaggle/input/**/config.json', recursive=True)]
    MODEL = os.path.dirname(hits[0]) if hits else 'google/gemma-2-2b-it'
print('checkpoint:', MODEL)
def cuda_runtime_usable() -> bool:
    if not torch.cuda.is_available():
        return False
    try:
        # is_available() alone was a false positive on one Kaggle image:
        # PyTorch saw the GPU, but its kernels did not support that device.
        probe = torch.arange(8, device='cuda')
        assert int((probe + 1).sum().item()) == 36
        return True
    except Exception as exc:
        print(f'CUDA runtime probe failed ({type(exc).__name__}: {exc}); using CPU')
        return False
DEVICE = 'cuda' if cuda_runtime_usable() else 'cpu'
DTYPE = torch.float16 if DEVICE == 'cuda' else torch.float32
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=DTYPE).to(DEVICE).eval()
print(f'loaded OK on {DEVICE} ({DTYPE})')

In [ ]:
@torch.no_grad()
def surprise(setup: str, punchline: str) -> dict:
    """Mean NLL of the punchline tokens given the setup. Deterministic."""
    ctx  = tok(setup + '\n', return_tensors='pt').to(DEVICE)
    full = tok(setup + '\n' + ' ' + punchline, return_tensors='pt').to(DEVICE)
    n_ctx = ctx.input_ids.shape[1]
    logits = model(**full).logits[0]
    tgt = full.input_ids[0][1:]
    lp = torch.log_softmax(logits[:-1].float(), dim=-1)
    positions = torch.arange(tgt.numel(), device=tgt.device)
    nll = -lp[positions, tgt][n_ctx - 1:]
    return {'S': nll.mean().item(), 'n_tokens': int(nll.numel())}

# The pinned reference is a SPECIFIC joke. An earlier version of this cell
# used a different speed-bump joke and then compared it to 3.19, which is
# not an instrument check at all — it is a fabricated one. The text below
# is the canonical `speed_bumps` reference, and the pinned run recorded 10
# punchline tokens, so the token count is checked too.
PINNED_SETUP = 'I told my therapist about my fear of speed bumps.'
PINNED_PUNCH = "She said I'm slowly getting over it."
PINNED_S, PINNED_TOKENS = 3.19, 10
r = surprise(PINNED_SETUP, PINNED_PUNCH)
print(f"speed_bumps   S = {r['S']:.3f}   over {r['n_tokens']} tokens")
print(f'pinned across 4 prior instruments: S = {PINNED_S} over {PINNED_TOKENS} tokens')
ok = abs(r['S'] - PINNED_S) < 0.35
print('instrument agreement:', 'YES' if ok else 'NO — distrust everything below')
if r['n_tokens'] != PINNED_TOKENS:
    print(f"  note: tokenised to {r['n_tokens']} tokens, not {PINNED_TOKENS} — "
          'a different tokenisation makes the S comparison approximate')

### Does style change what Gemma finds surprising?

This is the question the style axis was built to make askable. Each arm is a different machine for building an expectation; a **proverb control** is included because proverbs are short assertions with no punchline at all, so if there is any effect they should sit apart from the joke forms rather than among them.

Sampling is deterministic — items are ordered by a hash of their own text, so the same corpus always yields the same sample and the selection cannot track source or length.

**What this cannot say:** S is model surprisal, not funniness. These items carry no human grade. A style with higher S is not funnier; it is less predictable to a 2B model.

**And what the wider run found.** A fuller version of this study (8 items per arm, 88 measurements, bootstrap 95% CIs) produced the same rough ordering — `what_do_you_call` highest at 6.68, `knock_knock` near the bottom at 3.73 — but **0 of 10 arms had a confidence interval strictly above the proverb control's**. Every interval overlapped.

So the ordering is a hypothesis worth more data, not a result. The cell below therefore compares intervals rather than ranking means, because ranking means at this sample size would manufacture a separation the data does not contain.

In [ ]:
import hashlib, statistics
PER_ARM = 6   # small so the notebook finishes; the repo runs a wider sweep

arms = {}
for rec in st.iter_corpus():
    t = ' '.join(rec.get('text', '').split())
    if not (25 <= len(t) <= 220):
        continue
    m = rec.get('meta', {}) or {}
    if (m.get('language') or rec.get('language') or 'en') != 'en':
        continue
    cat = m.get('style_category')
    if cat in ('military', 'medical', 'legal', 'jest_book', 'riddles'):
        arms.setdefault('src:' + cat, []).append(t)
    if m.get('style') == 'dad_joke':
        arms.setdefault('src:dad_joke', []).append(t)
    if m.get('record_kind') == 'proverb':
        arms.setdefault('control_proverb', []).append(t)
    f = st.classify_form(t, m)['form']
    if f in ('knock_knock', 'what_do_you_call', 'walks_into_bar', 'chuck_norris'):
        arms.setdefault('form:' + f, []).append(t)

for k in arms:
    arms[k] = sorted(arms[k], key=lambda s: hashlib.sha256(s.encode()).hexdigest())[:PER_ARM]
for k, v in sorted(arms.items()):
    print(f'  {k:<24} {len(v):>3}')

In [ ]:
from mesh_signals import split_setup_punchline
results = {}
for arm, items in sorted(arms.items()):
    vals = []
    for t in items:
        s, p = split_setup_punchline(t)
        if s.strip() and p.strip():
            vals.append(surprise(s, p)['S'])
    if vals:
        results[arm] = vals

print(f"{'arm':<24} {'n':>3} {'mean S':>8} {'median':>8}")
for arm, v in sorted(results.items(), key=lambda kv: -statistics.mean(kv[1])):
    print(f'{arm:<24} {len(v):>3} {statistics.mean(v):>8.3f} {statistics.median(v):>8.3f}')

# Compare with UNCERTAINTY, not with bare means. Ranking means and saying
# 'every joke arm sits above the control' implies a separation that n=6
# cannot support; a wider run of this same study (n=8/arm, 88 measurements)
# found 0 of 10 arms separated once bootstrap CIs were computed.
import random
def boot_ci(xs, n=2000, seed=7):
    if len(xs) < 2: return (float('nan'), float('nan'))
    rng = random.Random(seed)
    ms = sorted(statistics.mean([xs[rng.randrange(len(xs))] for _ in xs]) for _ in range(n))
    return ms[int(0.025*n)], ms[int(0.975*n)]

ctrl = results.get('control_proverb')
if ctrl:
    c_lo, c_hi = boot_ci(ctrl)
    print(f'\nproverb control: mean {statistics.mean(ctrl):.3f}  95% CI [{c_lo:.3f}, {c_hi:.3f}]')
    sep = [a for a, v in results.items()
           if a != 'control_proverb' and boot_ci(v)[0] > c_hi]
    print(f'arms whose CI lies STRICTLY ABOVE the control CI: '
          f'{len(sep)}/{len(results)-1}')
    if not sep:
        print('  -> none. The ordering is suggestive; the SEPARATION IS NOT ESTABLISHED')
        print('     at this sample size. Reporting the ranking as a finding would')
        print('     be an overclaim.')
    else:
        print('  ->', sorted(sep))

## 5. What the human label can support

The largest source family is useful for a different reason: its captions carry the raw vote breakdown. That exposes label noise instead of pretending the crowd mean is perfect truth. The full grouped run is intentionally not repeated in this Kaggle notebook: it fits five contest-held-out models over 215,465 captions and is the slow step in the release. Its complete JSON receipt is versioned in the cloned repository and checked below against the two independent bound receipts.

The result has three denominators:

- **0.826 label ceiling:** finite votes limit agreement with the published mean.
- **0.411 text-only bound:** the same words travel poorly between drawings, so most caption reception is contextual.
- **held-out-contest score:** what 30 structural text features recover on drawings the model never saw. The completed grouped run reaches **rho = 0.1555**, or **37.8%** of the text-only bound; 358 of 360 contest-level correlations are positive. The old random split reads 0.1541 versus 0.1538 for held-out contests when both are pooled, so shared contest identity did not inflate this model's pooled score.

This section makes the expensive result visible without disguising a stored receipt as an in-notebook training run.

In [ ]:
from math import isclose
RECEIPTS = Path('jestry_out')
def receipt(name):
    path = RECEIPTS / name
    assert path.exists(), f'missing versioned receipt: {path}'
    return json.loads(path.read_text(encoding='utf-8'))

ceiling_receipt = receipt('caption_ceiling.json')
portability_receipt = receipt('caption_portability.json')
caption_model_receipt = receipt('caption_model.json')
label_ceiling = ceiling_receipt['headline']['median_ceiling']
text_bound = portability_receipt['results']['text_only_predictor_bound']
model_result = caption_model_receipt['results']['within_contest_median_spearman']
bounds = caption_model_receipt['bounds']
assert caption_model_receipt['receipt_version'] >= 2
assert caption_model_receipt['status'] == 'complete'
assert isclose(bounds['label_ceiling'], label_ceiling, abs_tol=1e-12)
assert isclose(bounds['text_only_bound'], text_bound, abs_tol=1e-12)
assert caption_model_receipt['n_contests'] >= 300
assert 'GroupKFold over contests' in caption_model_receipt['protocol']['cv']
print(f"label ceiling                 {label_ceiling:+.4f}")
print(f"text-only bound                {text_bound:+.4f}")
print(f"held-out-contest model         {model_result:+.4f}")
print(f"achieved / text-only bound     {bounds['achieved_over_text_only_bound']:.1%}")
print(f"captions / held-out contests   {caption_model_receipt['n_rows']:,} / "
      f"{caption_model_receipt['n_contests']}")
print(f"end-to-end model runtime       {caption_model_receipt['runtime_s']/60:.1f} min")
print('protocol receipt: PASS')

In [ ]:
import matplotlib.pyplot as plt
names = ['label ceiling', 'text-only bound', 'held-out model']
values = [label_ceiling, text_bound, model_result]
colors = ['#3987e5', '#d95926', '#199e70']
fig, ax = plt.subplots(figsize=(8, 3.6))
bars = ax.barh(names[::-1], values[::-1], color=colors[::-1])
ax.set_xlim(min(0, min(values) - .03), max(values) * 1.12)
ax.set_xlabel('Spearman correlation')
ax.set_title('Observed model result against achievable bounds', loc='left', weight='bold')
ax.axvline(0, color='#777777', linewidth=.8)
for bar, value in zip(bars, values[::-1]):
    ax.text(value + .012, bar.get_y() + bar.get_height()/2, f'{value:.3f}', va='center')
for spine in ('top', 'right', 'left'):
    ax.spines[spine].set_visible(False)
plt.tight_layout(); plt.show()

## What this is, and is not

**Is:** a research instrument spanning millions of items, dozens of language labels and hundreds of sources, with every row carrying its own source and licence, three style axes, and a deterministic Gemma measurement protocol.

**Is not:** a single-licence dataset. The live census above reports the exact research-only, noncommercial, redistributable and unclassified counts. A redistributor must honour the per-record `license` field, not a collection-level claim.

### Limits, stated rather than buried

1. **Concentration.** One source family is the majority of the full corpus, so any corpus-wide average describes it. The published slice is stratified per family specifically to break that, which is why its distribution differs from the full corpus by design.
2. **Form coverage.** Only a single-digit percentage of items receive a *specific* form label. The rest sit in buckets describing shape, not mechanism.
3. **Screening.** Content screening is a slur regex plus two hand-excluded volumes. It does not catch stereotype humor that contains no slurs, and the corpus demonstrably contains such items — an accent-based pun and a Gestapo light-bulb joke both surfaced in a random sample. A stereotype-level screen is a different and unsolved problem.
4. **S is not funniness.** Nothing in this notebook measures whether a joke is good. It measures how surprised a 2B model is by an ending.
5. **Domain labels are lexical guesses.** A joke mentioning a doctor is not necessarily a medical joke.

### Bugs found by checking rather than assuming

Recorded because each was silently wrong rather than loudly broken:

- An 8-character minimum deleted most Chinese content; a chengyu is four characters.
- A directory glob folded derived sidecar files into the corpus, inflating counts.
- `language` was read from one of its two possible locations, so exported rows read `unknown` — a valid-looking value.
- The limerick detector matched a bare apostrophe as a word, rejecting real limericks.
- Concurrent HuggingFace pulls rate-limit each other into silent truncation: an 8,000-row split first came back as 3,195.
- A study wrote its receipt only at the end, so a 90-minute run that hit its timeout produced nothing. It now checkpoints each measurement as it lands.

Full source inventory, dead links and parser debt: `SOURCE_SWEEP_2026-07-26.md`. Theory and falsifiable predictions: `THEORY.md`. Charter: `JESTRY-CHARTER-AND-CONSTITUTION-2026-07-23.md`.